# Interpretabilite et Analyse des Biais
## LIME, SHAP et considerations ethiques

**Objectif** : Comprendre *pourquoi* le modele prend ses decisions et identifier les biais potentiels.

**Plan** :
1. **LIME** : Explications locales (quels mots influencent chaque prediction)
2. **SHAP** : Importance globale des features
3. **Analyse de fairness** : Biais lies aux speakers, partis et sujets
4. **Discussion ethique** : Limites et risques de la detection automatique de fake news

**Outils** :
- `lime` : Local Interpretable Model-agnostic Explanations (Ribeiro et al., 2016)
- `shap` : SHapley Additive exPlanations (Lundberg & Lee, 2017)

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.metrics import accuracy_score, f1_score, classification_report
from lime.lime_text import LimeTextExplainer
import shap

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/Traitees")
MODELS_DIR = Path("../models")

print("Imports OK")

Imports OK


In [2]:
# Chargement des donnees et du modele
df_test = pd.read_parquet(DATA_DIR / "liar_test.parquet")
df_full = pd.read_parquet(DATA_DIR / "liar_complet.parquet")

X_test = df_test["statement_clean"]
y_test = df_test["label_binary"]

# Charger le modele TF-IDF + LogReg (pipeline complet)
model = joblib.load(MODELS_DIR / "tfidf_logreg_binary.joblib")
print(f"Modele charge : TF-IDF + LogReg")
print(f"Test set : {len(X_test)} declarations")

Modele charge : TF-IDF + LogReg
Test set : 1267 declarations


## 1. LIME — Explications locales

LIME perturbe le texte d'entree et observe comment les predictions changent pour identifier les mots les plus influents dans chaque decision. C'est une methode **model-agnostic** : elle fonctionne avec n'importe quel classifieur.

In [3]:
# Initialisation LIME
explainer = LimeTextExplainer(class_names=["Fake", "Real"], random_state=42)

# Fonction de prediction compatible LIME
def predict_proba(texts):
    """Wrapper pour que LIME puisse appeler notre pipeline."""
    return model.predict_proba(texts)

# Expliquer quelques predictions correctes et incorrectes
y_pred = model.predict(X_test)
correct_mask = y_pred == y_test.values
incorrect_mask = ~correct_mask

# Selectionner des exemples interessants
examples = {
    "Correct — Fake detecte": df_test[correct_mask & (y_test == 0)].iloc[0],
    "Correct — Real detecte": df_test[correct_mask & (y_test == 1)].iloc[0],
    "Erreur — Fake predit Real": df_test[incorrect_mask & (y_test == 0)].iloc[0] if incorrect_mask.any() else None,
    "Erreur — Real predit Fake": df_test[incorrect_mask & (y_test == 1)].iloc[0] if incorrect_mask.any() else None,
}

for title, row in examples.items():
    if row is None:
        continue
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"Texte : {row['statement'][:100]}...")
    print(f"Label reel : {'Fake' if row['label_binary'] == 0 else 'Real'}")
    
    exp = explainer.explain_instance(
        row["statement_clean"], predict_proba, num_features=10, num_samples=500
    )
    
    # Afficher les mots influents
    print("\nMots les plus influents :")
    for word, weight in exp.as_list():
        direction = "→ Fake" if weight < 0 else "→ Real"
        print(f"  {word:20s} {weight:+.4f}  {direction}")


Correct — Fake detecte
Texte : Says John McCain has done nothing to help the vets....
Label reel : Fake

Mots les plus influents :
  mccain               +0.0346  → Real
  says                 -0.0282  → Fake
  nothing              -0.0235  → Fake
  to                   -0.0218  → Fake
  has                  +0.0218  → Real
  john                 +0.0108  → Real
  the                  +0.0078  → Real
  done                 +0.0038  → Real
  help                 +0.0034  → Real
  vets                 +0.0005  → Real

Correct — Real detecte
Texte : Over the past five years the federal government has paid out $601 million in retirement and disabili...
Label reel : Real

Mots les plus influents :
  million              +0.0263  → Real
  government           -0.0193  → Fake
  to                   -0.0136  → Fake
  in                   +0.0125  → Real
  has                  +0.0106  → Real
  federal              +0.0086  → Real
  five                 +0.0085  → Real
  the                  +

## 2. SHAP — Importance globale des features

SHAP calcule la contribution de chaque feature (mot TF-IDF) a la prediction en utilisant la theorie des jeux de Shapley. On visualise les features les plus importantes a l'echelle globale.

In [4]:
# SHAP avec LinearExplainer (optimise pour modeles lineaires)
# Extraire le vectorizer et le classifieur du pipeline
tfidf = model.named_steps["tfidf"]
clf = model.named_steps["clf"]

# Transformer le test set
X_test_tfidf = tfidf.transform(X_test)

# SHAP Explainer pour modele lineaire
shap_explainer = shap.LinearExplainer(clf, X_test_tfidf, feature_perturbation="interventional")
shap_values = shap_explainer.shap_values(X_test_tfidf)

# Noms des features
feature_names = np.array(tfidf.get_feature_names_out())

print(f"SHAP values shape : {shap_values.shape}")
print(f"Nombre de features : {len(feature_names)}")

SHAP values shape : (1267, 7928)
Nombre de features : 7928


In [5]:
# Top features par importance SHAP moyenne absolue
mean_shap = np.abs(shap_values).mean(axis=0)

# Convertir sparse en array si necessaire
if hasattr(mean_shap, "A"):
    mean_shap = np.asarray(mean_shap).flatten()
else:
    mean_shap = np.asarray(mean_shap).flatten()

top_k = 30
top_idx = np.argsort(mean_shap)[-top_k:]

fig = px.bar(
    x=mean_shap[top_idx],
    y=feature_names[top_idx],
    orientation="h",
    title=f"Top {top_k} features par importance SHAP moyenne",
    labels={"x": "|SHAP value| moyen", "y": "Feature"},
    color=mean_shap[top_idx],
    color_continuous_scale="Reds",
)
fig.update_layout(template="plotly_white", height=600, coloraxis_showscale=False)
fig.show()

In [6]:
# Direction SHAP : quels mots poussent vers Fake vs Real
mean_shap_signed = np.asarray(shap_values.mean(axis=0)).flatten()

# Top mots poussant vers Real (positif)
top_real_idx = np.argsort(mean_shap_signed)[-15:]
# Top mots poussant vers Fake (negatif)
top_fake_idx = np.argsort(mean_shap_signed)[:15]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Poussent vers FAKE", "Poussent vers REAL"])

fig.add_trace(go.Bar(
    y=feature_names[top_fake_idx], x=mean_shap_signed[top_fake_idx],
    orientation="h", marker_color="#e74c3c",
), row=1, col=1)

fig.add_trace(go.Bar(
    y=feature_names[top_real_idx], x=mean_shap_signed[top_real_idx],
    orientation="h", marker_color="#2ecc71",
), row=1, col=2)

fig.update_layout(
    title="Direction SHAP : mots poussant vers Fake vs Real",
    template="plotly_white", height=500, showlegend=False,
)
fig.show()

## 3. Analyse de Fairness — Biais par speaker et parti politique

Un risque majeur de ce type de modele est qu'il apprenne a associer certains **speakers** ou **partis** a la desinformation plutot que d'analyser le contenu factuel. On mesure ici la performance du modele par sous-groupe.

In [7]:
# Performance par parti politique
df_test_eval = df_test.copy()
df_test_eval["prediction"] = model.predict(X_test)
df_test_eval["correct"] = df_test_eval["prediction"] == df_test_eval["label_binary"]

# Accuracy par parti
party_perf = []
for party in ["republican", "democrat"]:
    sub = df_test_eval[df_test_eval["party"] == party]
    if len(sub) < 10:
        continue
    acc = accuracy_score(sub["label_binary"], sub["prediction"])
    f1 = f1_score(sub["label_binary"], sub["prediction"], average="weighted")
    fake_rate_real = (sub["label_binary"] == 0).mean()
    fake_rate_pred = (sub["prediction"] == 0).mean()
    
    party_perf.append({
        "Parti": party.capitalize(),
        "N": len(sub),
        "Accuracy": round(acc, 4),
        "F1": round(f1, 4),
        "Taux Fake reel": round(fake_rate_real, 4),
        "Taux Fake predit": round(fake_rate_pred, 4),
        "Ecart Fake": round(fake_rate_pred - fake_rate_real, 4),
    })

party_df = pd.DataFrame(party_perf)
print("=== Performance par parti politique ===")
display(party_df)

# Interpretation
print("\nSi l'ecart Fake (predit - reel) est positif, le modele sur-predit les Fake pour ce parti.")
print("Si negatif, il sous-predit les Fake.")

=== Performance par parti politique ===


,Parti,N,Accuracy,F1,Taux Fake reel,Taux Fake predit,Ecart Fake
0,Republican,571,0.6077,0.6079,0.4623,0.5324,0.0701
1,Democrat,406,0.5837,0.5956,0.3300,0.4606,0.1305



Si l'ecart Fake (predit - reel) est positif, le modele sur-predit les Fake pour ce parti.
Si negatif, il sous-predit les Fake.


In [8]:
# Visualisation fairness par parti
fig = go.Figure()

for _, row in party_df.iterrows():
    fig.add_trace(go.Bar(
        name=row["Parti"],
        x=["Accuracy", "F1-Score", "Taux Fake reel", "Taux Fake predit"],
        y=[row["Accuracy"], row["F1"], row["Taux Fake reel"], row["Taux Fake predit"]],
    ))

fig.update_layout(
    barmode="group",
    title="Analyse de fairness par parti politique",
    yaxis_title="Score / Taux",
    template="plotly_white", height=450,
    yaxis=dict(range=[0, 1]),
)
fig.show()

In [9]:
# Biais par speaker : les top speakers ont-ils des performances differentes ?
top_speakers = df_full["speaker"].value_counts().head(10).index.tolist()

speaker_perf = []
for sp in top_speakers:
    sub = df_test_eval[df_test_eval["speaker"] == sp]
    if len(sub) < 5:
        continue
    acc = accuracy_score(sub["label_binary"], sub["prediction"])
    speaker_perf.append({"Speaker": sp, "N_test": len(sub), "Accuracy": round(acc, 4)})

if speaker_perf:
    sp_df = pd.DataFrame(speaker_perf).sort_values("Accuracy")
    
    fig = px.bar(
        sp_df, x="Accuracy", y="Speaker", orientation="h",
        color="Accuracy", color_continuous_scale="RdYlGn",
        text="N_test",
        title="Accuracy par speaker (top speakers du dataset)",
        labels={"N_test": "Nb test"},
    )
    fig.update_traces(texttemplate="n=%{text}", textposition="outside")
    fig.update_layout(template="plotly_white", height=400, coloraxis_showscale=False,
                      xaxis=dict(range=[0, 1]))
    fig.show()
    
    print("\nAttention : des differences d'accuracy par speaker peuvent indiquer que")
    print("le modele a memorise des patterns specifiques a certains locuteurs.")


Attention : des differences d'accuracy par speaker peuvent indiquer que
le modele a memorise des patterns specifiques a certains locuteurs.


In [10]:
# Taux d'erreur par speaker (top 20 speakers les plus frequents dans tout le dataset)
# Cela revele si le modele sur-apprend certains locuteurs
error_by_speaker = (
    df_test_eval.groupby("speaker")
    .apply(lambda g: pd.Series({
        "n": len(g),
        "error_rate": (g["prediction"] != g["label_binary"]).mean(),
        "fake_rate_real": (g["label_binary"] == 0).mean(),
    }))
    .query("n >= 3")  # au moins 3 declarations pour etre significatif
    .sort_values("error_rate", ascending=False)
    .head(20)
)

print("=== Top 20 speakers par taux d'erreur (min 3 declarations) ===")
print(error_by_speaker.round(3).to_string())

fig = px.bar(
    error_by_speaker.reset_index(),
    x="error_rate", y="speaker", orientation="h",
    color="error_rate", color_continuous_scale="Reds",
    title="Taux d'erreur du modele par speaker (top 20)",
    labels={"error_rate": "Taux d'erreur", "speaker": "Speaker"},
)
fig.update_layout(template="plotly_white", height=500, coloraxis_showscale=False,
                  yaxis=dict(autorange="reversed"))
fig.show()

print("\nSi certains speakers ont un taux d'erreur tres different de la moyenne,")
print("cela peut indiquer que le modele a memorise leur style plutot que le contenu.")

=== Top 20 speakers par taux d'erreur (min 3 declarations) ===
                                     n  error_rate  fake_rate_real
speaker                                                           
dennis-kucinich                    3.0       1.000           0.333
george-lemieux                     3.0       1.000           0.333
john-oliver                        3.0       1.000           0.000
john-kitzhaber                     6.0       0.833           0.167
rob-portman                        4.0       0.750           0.250
bill-richardson                    4.0       0.750           0.250
chris-abele                        3.0       0.667           0.667
julian-castro                      3.0       0.667           0.000
mitch-mcconnell                    3.0       0.667           0.333
jeanne-shaheen                     3.0       0.667           0.333
herman-cain                        3.0       0.667           0.333
greater-wisconsin-political-fund   3.0       0.667           0.333


Si certains speakers ont un taux d'erreur tres different de la moyenne,
cela peut indiquer que le modele a memorise leur style plutot que le contenu.


## 4. Analyse des biais et considerations ethiques

### 4.1 Sources de biais identifiees

| Type de biais | Description | Impact |
|--------------|-------------|--------|
| **Biais de source** | Le LIAR dataset provient uniquement de PolitiFact (US) | Le modele est biaise vers la politique americaine |
| **Biais de speaker** | Certains speakers ont plus de declarations | Le modele peut memoriser le style d'un speaker |
| **Biais partisan** | Desequilibre entre partis dans le dataset | Le modele peut penaliser un parti plus que l'autre |
| **Biais d'annotation** | Les labels dependent de fact-checkers humains | Subjectivite dans le mapping des 6 classes |
| **Biais linguistique** | Dataset en anglais uniquement | Inapplicable a d'autres langues |
| **Biais temporel** | Declarations d'une epoque donnee | Vocabulaire et sujets obsoletes pour du contenu recent |

### 4.2 Le modele peut-il favoriser ou penaliser certains groupes ?

**Oui, clairement.** Notre analyse montre que :
- L'accuracy varie entre partis politiques → un parti peut etre systematiquement plus penalise
- Les top speakers ont des accuracy tres differentes → le modele peut avoir appris leur "style"
- Les mots importants (SHAP) incluent des termes specifiques a certains sujets politiques

### 4.3 Les erreurs pourraient-elles avoir des consequences en contexte reel ?

**Oui, potentiellement graves** :
- **Faux positifs** (Real → Fake) : Censurer des declarations veridiques = atteinte a la liberte d'expression
- **Faux negatifs** (Fake → Real) : Laisser passer de la desinformation = echec de la mission
- Dans un systeme automatise de moderation, chaque erreur a un cout social

### 4.4 Peut-on detecter la desinformation uniquement a partir du texte ?

**Non, pas de maniere fiable** :
- Le texte seul ne contient pas les informations factuelles necessaires pour verifier une declaration
- Le modele detecte des **patterns stylistiques** (ton, vocabulaire) correles avec la desinformation
- Une declaration factuelle peut utiliser un vocabulaire "suspect" et vice versa
- La detection de fake news est un probleme qui necessite **verification factuelle**, **contexte** et **jugement humain**

Le modele NLP est un **outil d'aide a la decision**, pas un juge de verite.

## 5. Synthese

**Ce que l'interpretabilite nous apprend** :
1. **LIME** revele que le modele s'appuie sur des mots-cles politiques specifiques plutot que sur une comprehension factuelle
2. **SHAP** montre que certains termes ont une importance disproportionnee — signe de biais lexical
3. **L'analyse de fairness** confirme que la performance varie entre sous-groupes (partis, speakers)

**Recommandations** :
- Ne pas deployer le modele comme outil de censure automatique
- Utiliser les explications LIME/SHAP pour auditer les decisions
- Combiner le modele avec des sources de fact-checking humain
- Reentrainer regulierement sur des donnees recentes pour eviter le biais temporel
- Monitorer les metriques de fairness entre sous-groupes en production